In [2]:
# ============== ASOSIY KOD - BU KODNI KAGGLE'GA PASTE QILING ==============

import os, random, numpy as np
import torch, torch.nn as nn, torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
from torch.cuda.amp import autocast, GradScaler
import shutil
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

# ============== 1. DATASET TAYYORLASH ==============
def prepare_dataset():
    """Kaggle datasetini train/val ga bo'lish"""
    base_dir = "flowers"
    output_dir = "/kaggle/working/flowers_split"
    
    print("Dataset tayyorlanmoqda...")
    
    for split in ['train', 'val']:
        os.makedirs(f"{output_dir}/{split}", exist_ok=True)
    
    for class_name in os.listdir(base_dir):
        class_path = os.path.join(base_dir, class_name)
        if not os.path.isdir(class_path):
            continue
            
        os.makedirs(f"{output_dir}/train/{class_name}", exist_ok=True)
        os.makedirs(f"{output_dir}/val/{class_name}", exist_ok=True)
        
        images = [img for img in os.listdir(class_path) if img.endswith(('.jpg', '.png', '.jpeg'))]
        train_imgs, val_imgs = train_test_split(images, test_size=0.2, random_state=42)
        
        print(f"{class_name}: train={len(train_imgs)}, val={len(val_imgs)}")
        
        for img in train_imgs:
            src = f"{class_path}/{img}"
            dst = f"{output_dir}/train/{class_name}/{img}"
            shutil.copy(src, dst)
            
        for img in val_imgs:
            src = f"{class_path}/{img}"
            dst = f"{output_dir}/val/{class_name}/{img}"
            shutil.copy(src, dst)
    
    print(f"Dataset tayyor: {output_dir}")
    return output_dir

# Dataset tayyorlash
DATA_DIR = prepare_dataset()

# ============== 2. HYPERPARAMETERS ==============
NUM_CLASSES = 5
BATCH = 32
EPOCHS_HEAD = 3   # Faqat head o'qitish
EPOCHS_FULL = 7   # Butun model o'qitish
EPOCHS_RES = 10   # ResNet uchun
LR_HEAD = 3e-4
LR_FULL = 1e-4
LR_RES = 3e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
NUM_WORKERS = 2
USE_AMP = True

print(f"Device: {DEVICE}")

# ============== 3. SEED SOZLASH ==============
def set_seed(sd=SEED):
    random.seed(sd)
    np.random.seed(sd)
    torch.manual_seed(sd)
    torch.cuda.manual_seed_all(sd)

set_seed()

# ============== 4. DATA TRANSFORMS ==============
mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

val_tfms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# ============== 5. DATALOADERS ==============
train_ds = datasets.ImageFolder(f"{DATA_DIR}/train", transform=train_tfms)
val_ds = datasets.ImageFolder(f"{DATA_DIR}/val", transform=val_tfms)

train_ld = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_ld = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

idx_to_class = {v: k for k, v in train_ds.class_to_idx.items()}
print(f"Sinflar: {idx_to_class}")
print(f"Train: {len(train_ds)} ta rasm")
print(f"Val: {len(val_ds)} ta rasm")

# ============== 6. LOSS FUNCTION ==============
criterion = nn.CrossEntropyLoss()


Dataset tayyorlanmoqda...
daisy: train=611, val=153
dandelion: train=841, val=211
rose: train=627, val=157
sunflower: train=586, val=147
tulip: train=787, val=197
Dataset tayyor: /kaggle/working/flowers_split
Device: cpu
Sinflar: {0: 'daisy', 1: 'dandelion', 2: 'rose', 3: 'sunflower', 4: 'tulip'}
Train: 3452 ta rasm
Val: 865 ta rasm


In [3]:
# Pretrained ResNet18
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, NUM_CLASSES)
model = model.to(DEVICE)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\behru/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100.0%


In [ ]:
optimizer = optim.Adam(model.parameters(), lr=LR_FULL)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

In [5]:
scaler = GradScaler(enabled=USE_AMP)

C:\Users\behru\AppData\Local\Temp\ipykernel_19752\2860490094.py:1: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=USE_AMP)
c:\Users\behru\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\amp\grad_scaler.py:136: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(


In [6]:
def train_one_epoch(model, loader, criterion, optimizer, device, scaler):
    model.train()
    running_loss, correct = 0, 0
    total = 0
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        with autocast(enabled=USE_AMP):
            outputs = model(images)
            loss = criterion(outputs, labels)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc



In [7]:
def validate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0, 0, 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    val_loss = running_loss / total
    val_acc = correct / total
    return val_loss, val_acc


In [8]:
best_acc = 0
for epoch in range(EPOCHS_FULL):
    train_loss, train_acc = train_one_epoch(model, train_ld, criterion, optimizer, DEVICE, scaler)
    val_loss, val_acc = validate(model, val_ld, criterion, DEVICE)
    
    scheduler.step()
    
    print(f"Epoch {epoch+1}/{EPOCHS_FULL} | "
          f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}")
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "/kaggle/working/best_model.pth")
  

c:\Users\behru\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\behru\AppData\Local\Temp\ipykernel_19752\1693556972.py:10: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP):
c:\Users\behru\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\amp\autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


KeyboardInterrupt: 